# 长尾场景分类可视化监控逻辑
本笔记本实现了实时摄像头流的可视化，左侧显示实时画面，右侧显示模型处理的最新分析帧。

In [ ]:
import cv2
import ipywidgets.widgets as widgets
from IPython.display import display
import threading
import time
import os
import sys
import yaml
from datetime import datetime

# 配置路径
WORK_DIR = "/home/pi/Desktop/VehicleCloudCollaboration"
sys.path.append(WORK_DIR)
sys.path.append(os.path.join(WORK_DIR, 'utils'))

from classifier import LongTailClassifier
from McLumk_Wheel_Sports import move_forward, stop

# 加载配置
def load_config(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

config = load_config(os.path.join(WORK_DIR, 'config.yaml'))
classifier = LongTailClassifier(config)

# 初始化组件
image_stream = widgets.Image(format='jpeg', width=480, height=360)
image_analysis = widgets.Image(format='jpeg', width=480, height=360)
output_info = widgets.Output()

# 布局：左侧实时流，右侧分析帧
ui = widgets.HBox([
    widgets.VBox([widgets.Label("实时视频流 (Live)"), image_stream]),
    widgets.VBox([widgets.Label("上次分析帧 (Analysis)"), image_analysis])
])

# 控制组件
speed_slider = widgets.IntSlider(value=10, min=0, max=255, description='行驶速度:')
start_button = widgets.Button(description="启动监控", button_style='success')
stop_button = widgets.Button(description="停止监控", button_style='danger')

def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

# 全局状态
running = False

def camera_thread_func():
    global running
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    last_analysis_time = 0
    analysis_interval = 3.0 # 3秒检测一次
    
    # 初始启动小车
    move_forward(speed_slider.value)
    
    try:
        while running:
            ret, frame = cap.read()
            if not ret:
                continue
                
            # 更新实时流
            image_stream.value = bgr8_to_jpeg(frame)
            
            now = time.time()
            if now - last_analysis_time >= analysis_interval:
                # 执行分析
                temp_path = os.path.join(WORK_DIR, "captured_frames", "current_analysis.jpg")
                os.makedirs(os.path.dirname(temp_path), exist_ok=True)
                cv2.imwrite(temp_path, frame)
                
                result = classifier.predict(temp_path)
                
                # 在分析帧上绘制结果
                analysis_frame = frame.copy()
                status_text = "LONG-TAIL" if result['is_long_tail'] else "NORMAL"
                color = (0, 0, 255) if result['is_long_tail'] else (0, 255, 0)
                
                cv2.putText(analysis_frame, f"{status_text} ({result['score']:.2f})", 
                            (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
                
                image_analysis.value = bgr8_to_jpeg(analysis_frame)
                
                # 控制小车
                with output_info:
                    if result['is_long_tail']:
                        print(f"[{datetime.now().strftime('%H:%M:%S')}] 🛑 警告：检测到长尾！刹车。")
                        stop()
                    else:
                        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ 正常：继续以此速度前进：{speed_slider.value}")
                        move_forward(speed_slider.value)
                
                last_analysis_time = now
                
            time.sleep(0.01)
    finally:
        stop()
        cap.release()
        with output_info:
            print("摄像头已释放，监控停止。")

def start_click(b):
    global running
    if not running:
        running = True
        thread = threading.Thread(target=camera_thread_func)
        thread.start()
        with output_info:
            print("开始监控...")

def stop_click(b):
    global running
    running = False
    stop()
    with output_info:
        print("正在停止...")

start_button.on_click(start_click)
stop_button.on_click(stop_click)

display(widgets.VBox([ui, widgets.HBox([speed_slider, start_button, stop_button]), output_info]))